**[🏠 Course Home](../README.md)** | **📖 [Conceptual
Track](../conceptual/01_foundations_and_conjugate_updating.ipynb)** | **🤖
[AI-Native Track](01_foundations_and_conjugate_updating.ipynb)** | **🐍 [Python
Track](../python/01_foundations_and_conjugate_updating.ipynb)** | **🔵 [R
Track](../r/01_foundations_and_conjugate_updating.ipynb)** | **⬅️ [Prev: Ch
00](00_START_HERE.ipynb)** | **➡️ [Next: Ch
02](02_normal_conjugate_updating.ipynb)**

*🧭 Part I: Foundations, Conjugacy & Estimating a Mean — Chapter 01 of 22*

---

# Chapter 01: Discrete Beta-Binomial Conjugacy
### *Epistemic Auditing, Failure Rate Inference & Prompting Contracts*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> Discrete pass/fail outcomes are the primary sensory inputs of software
> systems: CI test suites, packet loss counters, RPC error budgets, and user
> conversions. When sample sizes are small ($n=20$) or failures are rare,
> unconstrained AI agents systematically produce dangerous statistical
> hallucinations—outputting negative probability bounds, ignoring prior domain
> stability, or collapsing uncertainty. This module establishes how to prompt
> an LLM as a rigorous probabilistic compiler using the exact Beta-Binomial
> conjugate model.

---


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

Consider what happens when an engineer asks an LLM for a failure rate analysis
after running a quick batch of integration tests.

### The Naive Prompt
```text
We ran 20 automated integration test runs on the new deployment candidate,
and 3 failed. What is our service failure rate, and what is the 95%
confidence interval for production?
```

### The Plausible-Sounding AI Hallucination
```text
Based on your test results (3 failures out of 20 runs):

1. Failure Rate:
   p_hat = 3 / 20 = 0.15 (15.0%)

2. Standard Error (using the binomial proportion formula):
   SE = sqrt(p * (1 - p) / n)
      = sqrt(0.15 * 0.85 / 20) = sqrt(0.006375) ≈ 0.0798 (7.98%)

3. 95% Confidence Interval (Z = 1.96):
   CI = 0.15 ± 1.96 * 0.0798
      = 0.15 ± 0.1565
      = [-0.0065, 0.3065] -> [-0.65%, 30.65%]

Conclusion:
There is a 95% probability that your service's true failure rate lies between
-0.65% and 30.65%. Because the failure rate could be as high as 30.65%,
you should hold the release.
```

---

### Forensic Post-Mortem: Why This AI Output is Catastrophic

```
   NAIVE AI OUTPUT: [-0.65%, 30.65%]
   ───[Negative Space]─────────[0.0%]══════════════════════════[30.65%]───▶ θ
      ▲ IMPOSSIBLE!             ▲ Physical Boundary
```

1. **Physical Boundary Violation ($\theta < 0$):**
   The AI computed a lower bound of $-0.65\%$. A failure rate is a probability
   parameter bounded on the simplex $\theta \in [0, 1]$. Reporting a negative
   probability is an immediate signature of uncalibrated asymptotic formulas.

2. **Asymptotic Breakdown Under Small Samples & Skew ($n p < 10$):**
   The AI used the Wald interval ($\hat{p} \pm 1.96
   \sqrt{\hat{p}(1-\hat{p})/n}$).
   The Central Limit Theorem requires $n p \ge 10$ and $n(1-p) \ge 10$ for the
   sampling distribution to approach a symmetric Gaussian. Here, $n p = 3$.
   The true binomial likelihood is heavily skewed to the right; forcing a
   symmetric bell curve onto it is mathematically invalid.

3. **Base-Rate Amnesia (Domain Prior Ignored):**
   The service in question did not appear out of thin air. Historical production
   telemetry shows a $90\%$ baseline pass rate ($10\%$ baseline failure rate)
   over thousands of runs. By computing raw Maximum Likelihood Estimation (MLE),
   the AI threw away all prior engineering knowledge, letting 20 noisy runs
   overrule months of system stability.

4. **The Frequentist Confidence Fallacy:**
   The AI claimed: *"There is a 95% probability that your service's true
   failure rate lies between -0.65% and 30.65%"*. In frequentist probability,
   the true parameter $\theta$ is a fixed, non-random constant. The true
   probability of $\theta$ being in any single observed interval is strictly
   $0$ or $1$. Only a Bayesian Credible Interval allows a direct probability
   statement about parameter truth.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: We observed 3 test failures in 20 canary runs.
2. Prior: Historically, healthy builds have a 90% pass rate (Beta(10, 90),
   equivalent to ~100 historical runs).
3. Economics: Leaking a broken build costs 20x more than a false rollback
   (C_leak / C_abort = 20).

Task: Run a conjugate Beta-Binomial update, report P(failure rate > 15%), and
tell me whether to quarantine based on expected cost.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Model:  k ~ Binomial(n=20, theta),  theta ~ Beta(10, 90)
Data:   k = 3
Action: Quarantine if E[Loss(ship)] > E[Loss(quarantine)] with C_leak/C_abort = 20
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Principal Bayesian Statistician. I had 3 test failures in 20 canary
> runs. Before recommending any analysis or action, ask me the 3 most critical
> questions about historical baselines, sample independence, and business costs
> to construct a calibrated conjugate model."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
probabilistic_contract:
  target: "service_failure_rate_inference"
  parameter:
    symbol: "theta"
    domain: [0.0, 1.0]
    interpretation: "Underlying physical failure probability per execution"
  observables:
    n_trials: 20
    k_failures: 3
    data_generating_process: "Binomial(n=20, theta)"
  prior_model:
    distribution: "Beta(alpha_0=10, beta_0=90)"
    rationale: "Historical baseline: 90% pass rate over N_prior=100"
    prior_mean: 0.1000
    prior_mode: 0.0918
    prior_variance: 0.000891
  likelihood_model:
    family: "Binomial"
    equation: "P(k=3 | n=20, theta) = binom(20, 3) * theta^3 * (1 - theta)^17"
  computational_invariants:
    conjugate_class: "Beta-Binomial"
    updating_equations:
      alpha_post: "alpha_0 + k = 10 + 3 = 13"
      beta_post: "beta_0 + (n - k) = 90 + 17 = 107"
    posterior_family: "Beta(alpha_post=13, beta_post=107)"
    effective_sample_size: "alpha_post + beta_post = 120"
  decision_loss_contract:
    loss_function: "Asymmetric SLA Threshold Loss"
    sla_critical_threshold: 0.15
    quarantine_rule: "TRIGGER if P(theta > 0.15 | data) > 0.05 else PASS"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an AI agent receives the Level 1 Micro-Spec from Part 2, it no longer
hallucinates impossible negative failure rates or relies on asymptotic bell
curves. Instead, it executes an exact conjugate update and produces the
following verified mathematical telemetry:

### Mathematical Derivation
1. **Prior Distribution**:
   $$p(\theta) = \frac{\theta^{10 - 1}(1 - \theta)^{90 - 1}}{\text{B}(10, 90)} = \frac{\theta^9 (1 - \theta)^{89}}{\text{B}(10, 90)}$$
2. **Likelihood Function**:
   $$p(k=3 \mid n=20, \theta) = \binom{20}{3} \theta^3 (1 - \theta)^{17}$$
3. **Conjugate Posterior**:
   $$p(\theta \mid D) \propto \theta^{9 + 3} (1 - \theta)^{89 + 17} = \theta^{12} (1 - \theta)^{106} \implies \text{Beta}(13, 107)$$

### Exact Posterior Telemetry
* **Posterior Mean**: $\mathbb{E}[\theta \mid D] = \frac{13}{120} \approx
  10.83\%$
* **Posterior Mode (MAP)**: $\frac{13 - 1}{13 + 107 - 2} = \frac{12}{118}
  \approx 10.17\%$
* **Posterior Standard Deviation**: $\sigma_{\text{post}} \approx 2.83\%$
* **95% Equal-Tailed Credible Interval**: $[5.94\%, 16.98\%]$
* **95% Highest Density Interval (HDI)**: $[5.51\%, 16.42\%]$

```
                POSTERIOR DENSITY: Beta(13, 107)
   Density
     ▲
14.0 ┤                 ***
12.0 ┤               **   **
10.0 ┤              *       *
 8.0 ┤             *         *
 6.0 ┤            *           *
 4.0 ┤           *             *
 2.0 ┤         **               **        [SLA Threshold: 15%]
 0.0 ┼────────*───────────────────*───────|──────────────▶ θ
    0.00     0.05               0.10     0.15    0.20   0.25
              ▲                  ▲        ▲
           2.5% CI            Mean/Mode  P(θ > 0.15) = 9.34%
           (5.94%)            (10.83%)   ==> TRIGGER QUARANTINE!
```

### Risk & Decision Evaluation
$$P(\theta > 0.15 \mid D) = 1 - \int_0^{0.15} \frac{\theta^{12}(1-\theta)^{106}}{\text{B}(13, 107)} \, d\theta \approx 0.0934 \quad (9.34\%)$$

Since $P(\theta > 0.15 \mid D) = 9.34\% > 5.00\%$, the candidate **fails the
quarantine decision rule**. The build is automatically quarantined for triage.


## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated response: it derived the posterior
$\text{Beta}(13, 107)$, plotted the density curve, and recommended a quarantine
action.

However, as an **Epistemic Architect**, you must never accept an AI agent's
mathematical claims at face value. How do you know the agent didn't quietly swap
$\alpha$ and $\beta$, miscalculate the tail risk integral, or hallucinate the
credible interval bounds?

Before deploying the agent's recommendation to production or letting an
autonomous CI bot block a pull request, you audit the agent against **four
mandatory verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Exact pseudo-count algebra & probability conservation on telemetry │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Exact Ingestion Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Domain Sanity Gate)

* **The Production Risk (Why This Gate Exists)**:
  In continuous integration and automated release testing, an unguided AI agent
  will frequently choose a mathematically naive "flat" prior like
  $\text{Beta}(1,
  1)$ without analyzing its generative consequences. For a canary suite of
  $n=20$
  runs, a $\text{Beta}(1, 1)$ prior asserts that observing $18$, $19$, or $20$
  failures is just as probable prior to testing as observing $0$ failures. In a
  production release gate, an uncalibrated prior that expects high failure
  counts
  will severely dilute genuine failure signals, allowing catastrophic build
  regressions to slip into production. Conversely, an overly tight prior will
  trigger spurious rollbacks. Gate 1 ensures the prior generates sane baseline
  realities before touching empirical data.
* **First-Principles Mathematical Mechanism**:
  We evaluate the marginal distribution of synthetic observations
  $k_{\text{sim}}$
  by integrating out the latent failure probability $\theta$:
  $$p(k) = \int_0^1 p(k \mid \theta) \, p(\theta) \, d\theta = \int_0^1
  \binom{n}{k} \theta^k (1-\theta)^{n-k} \frac{\theta^{\alpha_0 -
  1}(1-\theta)^{\beta_0 - 1}}{\text{B}(\alpha_0, \beta_0)} \, d\theta$$
  This yields the compound $\text{Beta-Binomial}(n=20, \alpha_0=10, \beta_0=90)$
  distribution:
  $$p(k) = \binom{n}{k} \frac{\text{B}(k + \alpha_0, n - k +
  \beta_0)}{\text{B}(\alpha_0, \beta_0)}$$
  For our historical prior representing a $90\%$ pass rate ($\theta_0 = 0.10$):
  * Expected synthetic failures: $\mathbb{E}[k] = n \frac{\alpha_0}{\alpha_0 +
    \beta_0} = 20 \times \frac{10}{100} = 2.0$.
  * Prior predictive variance: $\text{Var}(k) = n \frac{\alpha_0
    \beta_0}{(\alpha_0 + \beta_0)^2} \frac{\alpha_0 + \beta_0 + n}{\alpha_0 +
    \beta_0 + 1} = 20 \times 0.09 \times \frac{120}{101} \approx 2.1386$.
  We draw $10{,}000$ synthetic parameter samples $\theta^{(s)} \sim
  \text{Beta}(10, 90)$
  and simulate $k_{\text{sim}}^{(s)} \sim \text{Binomial}(20, \theta^{(s)})$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Discrete Support Invariant: $k_{\text{sim}}^{(s)} \in \{0, 1, \dots, 20\}$
    strictly for $100\%$ of draws (`[PASS]`; any value outside $\{0, \dots,
    20\}$ triggers `[FAIL]`).
  * Expected Failure Invariant: $|\bar{k}_{\text{sim}} - 2.000| \le 0.050$
    (`[PASS]`).
  * Extreme Burst Bound: $\hat{P}(k \ge 8) = \frac{1}{S}\sum_{s=1}^S
    \mathbb{I}(k_{\text{sim}}^{(s)} \ge 8) \le 0.0050$ (`[PASS]`; healthy
    baselines must not expect burst failures).
  * Boundary Probabilities: $P(k = 0) > 0.050$ and $P(k = 20) < 10^{-6}$
    (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Beta-Binomial Prior Predictive Check):
  Model: n = 20 trials, Prior theta ~ Beta(10, 90).
  Simulate 10,000 synthetic observations:
  1. Draw theta^(s) ~ Beta(10, 90).
  2. Draw k_sim^(s) ~ Binomial(20, theta^(s)).
  Verify that:
  - 100% of k_sim draws fall strictly in {0, 1, ..., 20}.
  - Empirical mean k_sim is 2.00 ± 0.05.
  - Tail probability P(k >= 8) < 0.005.
  Report [PASS] with simulated moments or [FAIL] with violations.
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Fake-Data Recovery

* **The Production Risk (Why This Gate Exists)**:
  When an AI agent writes code to update a Beta distribution, it frequently
  makes
  subtle algebraic mistakes: swapping successes and failures ($\alpha + n - k$
  instead of $\alpha + k$), misindexing the sample size, or using an
  approximation
  formula for the credible interval that degrades in the tails. In automated
  testing pipelines, an inverted updater would interpret failures as successes,
  progressively lowering the estimated failure rate as tests fail! Gate 2 audits
  the mathematical engine against ground truth across thousands of simulated
  realities.
* **First-Principles Mathematical Mechanism**:
  SBC tests whether the updater satisfies Bayes' self-consistency theorem:
  $$\int_0^1 p(\theta^*) \sum_{k=0}^n p(k \mid \theta^*) p(\theta \mid k) \,
  d\theta^* = p(\theta)$$
  The automated verification algorithm proceeds as follows:
  1. Sample a ground-truth parameter $\theta^* \sim \text{Beta}(10, 90)$.
  2. Generate synthetic test results: $k_{\text{fake}} \sim \text{Binomial}(20,
     \theta^*)$.
  3. Condition the analytical updater on $k_{\text{fake}}$:
     $$\theta \mid k_{\text{fake}} \sim \text{Beta}(10 + k_{\text{fake}}, 90 +
     20 - k_{\text{fake}})$$
  4. Compute the exact equal-tailed $95\%$ credible interval $[\theta_{0.025},
     \theta_{0.975}]$ using the inverse Beta CDF (regularized incomplete beta
     function):
     $$\theta_{0.025} = I_{0.025}^{-1}(\alpha_{\text{post}},
     \beta_{\text{post}}), \qquad \theta_{0.975} =
     I_{0.975}^{-1}(\alpha_{\text{post}}, \beta_{\text{post}})$$
  5. Check whether $\theta^* \in [\theta_{0.025}, \theta_{0.975}]$ and record
     hit $h_m = 1$, else $0$.
  6. Compute the CDF rank statistic $U_m = I_{\theta^*}(\alpha_{\text{post}},
     \beta_{\text{post}})$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Empirical Coverage Invariant: Across $M = 5{,}000$ independent trials,
    coverage $\hat{C} = \frac{1}{M}\sum h_m$ must satisfy $|\hat{C} - 0.9500|
    \le 0.0080$ ($[94.20\%, 95.80\%]$) (`[PASS]`).
  * Rank Uniformity Invariant: Kolmogorov-Smirnov test on $\{U_m\}_{m=1}^M$
    against $\text{Uniform}(0, 1)$ must yield $p_{\text{KS}} > 0.010$
    (`[PASS]`).
  * Recovery Bias Invariant: Mean estimation error $|\frac{1}{M}\sum
    (\mathbb{E}[\theta \mid k_{\text{fake}}] - \theta^*)| \le 0.0020$
    (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration on Beta-Binomial):
  Execute 5,000 Monte Carlo recovery trials:
  For m = 1 to 5,000:
    1. Draw theta* ~ Beta(10, 90).
    2. Draw k_fake ~ Binomial(20, theta*).
    3. Compute posterior Beta(10 + k_fake, 110 - k_fake).
    4. Compute 95% equal-tailed credible interval [q_0.025, q_0.975].
    5. Test if theta* is inside [q_0.025, q_0.975].
  Assert that empirical coverage is 95.0% ± 0.8% and recovery bias is < 0.002.
  Report [PASS] or [FAIL] with empirical coverage and max bias.
  ```

---

### Gate 3: Conjugate Invariant & Precision Audit

* **The Production Risk (Why This Gate Exists)**:
  Analytical Bayesian models are governed by exact algebraic conservation laws.
  An AI agent might implement an update using floating-point approximations,
  drop normalizers, or corrupt pseudo-count additions. If an updater violates
  sample-size conservation or allows posterior variance to increase after
  observing
  data, downstream automated gates will make unpredictable risk evaluations.
  Gate 3 proves algebraic and information-theoretic correctness before decisions
  are rendered.
* **First-Principles Mathematical Mechanism**:
  In conjugate Beta-Binomial updating, each observed data point contributes
  exactly one unit of pseudo-count to the distribution:
  * **Pseudo-Count Conservation**:
    $$\alpha_{\text{post}} + \beta_{\text{post}} = (\alpha_0 + k) + (\beta_0 + n
    - k) = \alpha_0 + \beta_0 + n = 10 + 90 + 20 = 120$$
  * **Posterior Mean as Convex Combination**:
    The posterior mean is an exact weighted average of the prior mean and sample
    mean:
    $$\mathbb{E}[\theta \mid D] = \frac{\alpha_0 + \beta_0}{\alpha_0 + \beta_0 +
    n} \left(\frac{\alpha_0}{\alpha_0 + \beta_0}\right) + \frac{n}{\alpha_0 +
    \beta_0 + n} \left(\frac{k}{n}\right) = \frac{100}{120}(0.100) +
    \frac{20}{120}(0.150) = \frac{13}{120} \approx 0.10833$$
  * **Strict Variance Contraction**:
    Posterior variance must contract monotonically upon observing data ($n >
    0$):
    $$\text{Var}_{\text{prior}} = \frac{\alpha_0 \beta_0}{(\alpha_0 +
    \beta_0)^2(\alpha_0 + \beta_0 + 1)} = \frac{900}{100^2 \times 101} \approx
    8.9109 \times 10^{-4}$$
    $$\text{Var}_{\text{post}} = \frac{\alpha_{\text{post}}
    \beta_{\text{post}}}{(\alpha_{\text{post}} +
    \beta_{\text{post}})^2(\alpha_{\text{post}} + \beta_{\text{post}} + 1)} =
    \frac{13 \times 107}{120^2 \times 121} = \frac{1391}{1742400} \approx 7.9832
    \times 10^{-4}$$
    $$\frac{\text{Var}_{\text{post}}}{\text{Var}_{\text{prior}}} = \frac{7.9832
    \times 10^{-4}}{8.9109 \times 10^{-4}} \approx 0.8959 < 1.0000$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Pseudo-Count Invariant: $(\alpha_{\text{post}} + \beta_{\text{post}}) -
    (\alpha_0 + \beta_0 + n) == 0$ strictly (`[PASS]`).
  * Exact Posterior Mean Invariant: $|\mu_{\text{post}} - \frac{13}{120}| <
    10^{-12}$ (`[PASS]`).
  * Strict Variance Reduction: $\text{Var}_{\text{post}} <
    \text{Var}_{\text{prior}}$ strictly (`[PASS]`).
  * Hyperparameter Positivity: $\alpha_{\text{post}} > 0$ and
    $\beta_{\text{post}} > 0$ strictly (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Conjugate Algebraic Invariants):
  Given prior Beta(10, 90) and observed data k = 3 failures in n = 20 runs:
  1. Verify total pseudo-count equality:
     alpha_post + beta_post == 100 + 20 == 120.
  2. Verify exact posterior mean: mu_post == 13 / 120 ≈ 0.108333.
  3. Verify variance reduction: Var_post / Var_prior < 1.0 (exact: ~0.8959).
  Assert exact algebraic equality. Report [PASS] or [FAIL].
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Decision Execution

* **The Production Risk (Why This Gate Exists)**:
  A model can be mathematically consistent yet fail to retrodict the data that
  created it, signalling model misspecification. Furthermore, an agent that
  relies
  on point estimates will note that the posterior mean ($10.83\%$) is
  comfortably
  below the $15.0\%$ SLA threshold and recommend shipping the release. However,
  leaking a broken build into production carries a catastrophic asymmetric cost
  ($C_{\text{leak}} = 20 \times C_{\text{abort}}$). By failing to integrate the
  full posterior tail risk above $15\%$, a naive agent makes a disastrous
  deployment
  decision. Gate 4 guarantees both predictive fit and risk-calibrated action.
* **First-Principles Mathematical Mechanism**:
  1. **Posterior Predictive Retrodiction**:
     We simulate replicated datasets $\tilde{k}^{(s)} \sim \text{Binomial}(20,
     \theta^{(s)})$
     where $\theta^{(s)} \sim \text{Beta}(13, 107)$. The posterior predictive
     distribution is $\text{Beta-Binomial}(20, 13, 107)$. We evaluate the
     posterior
     predictive p-value:
     $$p_{\text{retro}} = P(\tilde{k} \ge k_{\text{obs}} = 3 \mid D) =
     \sum_{j=3}^{20} \binom{20}{j} \frac{\text{B}(j + 13, 20 - j +
     107)}{\text{B}(13, 107)} \approx 0.352$$
     Because $0.025 \le 0.352 \le 0.975$, the observed count $k=3$ is completely
     congruent with the posterior predictive density.
  2. **Asymmetric Decision Theory Execution**:
     Let the action space be $\mathcal{A} = \{\text{Ship}, \text{Quarantine}\}$.
     Let the parameter space be partitioned into $\Theta_{\text{healthy}} = [0,
     0.15]$
     and $\Theta_{\text{broken}} = (0.15, 1.00]$.
     Cost matrix:
     * $L(\text{Quarantine}, \text{healthy}) = C_{\text{abort}} = 1$.
     * $L(\text{Quarantine}, \text{broken}) = C_{\text{abort}} = 1$.
     * $L(\text{Ship}, \text{healthy}) = 0$.
     * $L(\text{Ship}, \text{broken}) = C_{\text{leak}} = 20$.
     The economic indifference threshold is:
     $$p^* = \frac{C_{\text{abort}}}{C_{\text{leak}}} = \frac{1}{20} = 0.0500
     \quad (5.00\%)$$
     We compute the exact posterior tail probability of exceeding the $15\%$
     SLA:
     $$P(\theta > 0.15 \mid D) = 1 - I_{0.15}(13, 107) \approx 0.0886 \quad
     (8.86\%)$$
     Expected Loss comparison:
     $$\mathbb{E}[L(\text{Ship})] = 20 \times P(\theta > 0.15 \mid D) = 20
     \times 0.0886 = 1.772$$
     $$\mathbb{E}[L(\text{Quarantine})] = 1.000$$
     Since $\mathbb{E}[L(\text{Ship})] > \mathbb{E}[L(\text{Quarantine})]$ (or
     equivalently, $8.86\% > 5.00\%$), the Bayes-optimal action is
     **QUARANTINE_BUILD**.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Retrodictive Coverage Invariant: $k_{\text{obs}} = 3$ lies within the $95\%$
    predictive interval $[0, 5]$ (`[PASS]`).
  * Retrodictive P-Value Invariant: $0.10 \le p_{\text{retro}} \le 0.90$
    (`[PASS]`).
  * Exact Tail Risk Calculation: $|P(\theta > 0.15 \mid D) - 0.0886| \le 0.0020$
    (`[PASS]`).
  * Optimal Decision Invariant: `action == QUARANTINE_BUILD` strictly (`[PASS]`;
    any agent output recommending `SHIP` or `DEPLOY` triggers immediate
    `[FAIL]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Retrodiction & Asymmetric Decision Execution):
  Posterior: Beta(13, 107), Observed failures: k = 3 in n = 20 runs.
  1. Generate 10,000 posterior predictive replications:
     k_rep ~ Beta-Binomial(20, 13, 107).
  2. Verify observed k = 3 falls within the central 95% predictive interval.
  3. Compute exact tail risk P(theta > 0.15 | D) using regularized
     incomplete Beta.
  4. Compare tail risk against economic risk threshold p* = 1 / 20 = 0.0500.
  5. Select action minimizing expected loss.
  Assert action == QUARANTINE_BUILD. Report [PASS] or [FAIL].
  ```


## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Dimension | Naive Conversational Prompt | Calibrated Epistemic Specification |
| :--- | :--- | :--- |
| **Statistical Framework** | Frequentist Wald normal approximation | Bayesian Beta-Binomial conjugate updating |
| **Physical Support** | Unconstrained: $[-\infty, \infty]$ (yields negative rate: $-0.65\%$) | Strictly constrained simplex: $\theta \in [0, 1]$ |
| **Handling of Historical Data** | Base-rate neglect: treats 20 trials as sole source of truth | Bayesian prior: 100 historical pseudo-observations ($\text{Beta}(10, 90)$) |
| **Small-Sample Behavior ($n=20$)** | High variance, extreme skew distortion, asymptotic failure | Exact finite-sample posterior via Euler Beta integral |
| **Interval Meaning** | False claim: "95% probability true parameter is inside" | Mathematically true: $\int_{0.0594}^{0.1698} p(\theta \mid D) d\theta = 0.95$ |
| **Operational Output** | Ambiguous advice: "rate might be 30%, release held" | Deterministic loss contract: $P(\theta > 0.15) = 9.34\% > 5\% \implies$ Quarantine |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer on your team pastes this LLM output into an incident post-mortem:
> *"During our canary deployment, 1 out of 15 database write requests failed.
> Our AI diagnostic tool reports the failure rate is $6.67\%$ with a 95%
> confidence interval of $[-5.93\%, 19.27\%]$. Because zero is included in the
> confidence interval, there is no statistically significant evidence of a
> database problem."*

Identify the two statistical errors in this statement and explain why concluding
"no problem" is dangerous.

<details>
<summary><b>Click here to view the Tier 1 Solution...</b></summary>

1. **Physical Boundary Violation & Asymptotic Invalidity**:
   The interval contains negative failure rates ($-5.93\%$). Furthermore,
   $n \cdot p = 15 \times (1/15) = 1.0 \ll 10$, violating the minimal sample
   size required for a normal approximation.
2. **The "Zero Included in Interval" Absurdity**:
   In hypothesis testing for differences between two groups ($A - B$), zero
   indicates no difference. But here, the parameter is an **absolute failure
   rate $\theta \ge 0$**. Zero is the physical boundary of perfection! Claiming
   "zero is in the interval so there is no problem" is a complete category
   error—by that logic, a failure rate of $90\% \pm 95\%$ would also "include
   zero."
3. **Underestimated Upper Bound Risk**:
   The upper bound is nearly $20\%$. In a database tier, a $15\%$ or $20\%$
   write failure rate is a tier-1 outage. Under a neutral $\text{Beta}(1, 1)$
   prior, observing $1/15$ yields posterior $\text{Beta}(2, 15)$, with mean
   $2/17 \approx 11.76\%$ and $P(\theta > 0.05 \mid D) \approx 72.8\%$.
   There is substantial evidence of degradation.
</details>

---

### 🟡 Tier 2: Write an Everyday 3-Sentence Micro-Spec
An uncalibrated engineer prompts an AI:
```text
We tested a new checkout button. Out of 50 users, 45 converted. The old button
had an 80% conversion rate. Tell me if the new button is better.
```
Refactor this request into an everyday **3-sentence micro-spec** (Data, Prior,
Loss/Action threshold). Assume the 80% baseline reflects $N=100$ historical
trials, and business policy requires 95% certainty of improvement ($p^* = 0.95$)
to justify deployment.

<details>
<summary><b>Click here to view the Tier 2 Solution...</b></summary>

**1. The 3-Sentence Micro-Spec Prompt**:
```text
1. Data: We observed 45 conversions out of 50 users on the new button.
2. Prior: Baseline button conversion is Beta(80, 20) (80% rate, N=100).
3. Loss/Action: Promote new button only if P(theta > 0.80 | Data) >= 0.95.
```

**2. Mathematical Calculation**:
* Prior: $\text{Beta}(\alpha_0 = 80, \beta_0 = 20)$
* Data: $k = 45, \; n - k = 50 - 45 = 5$
* Posterior: $\text{Beta}(80 + 45, 20 + 5) = \text{Beta}(125, 25)$
* Posterior mean: $\mu_{\text{post}} = 125 / 150 \approx 83.33\%$
* Tail probability:
  $$P(\theta > 0.80 \mid D) = 1 - I_{0.80}(125, 25) \approx 86.22\%$$

**3. Resulting Action Decision**:
Because $86.22\% < 95.00\%$, the evidence fails the decision threshold.
**Action: HOLD / CONTINUE TESTING**. Do not promote the new button yet.
</details>

---

### 🔴 Tier 3: Audit an Autonomous Agent's Claims
An autonomous CI agent is deployed to evaluate canary test failures. Given a
baseline prior $\text{Beta}(5, 95)$ (5% historical failure rate) and observed
data of $k = 4$ failures in $n = 25$ canary executions, the agent posts:
> *"Posterior failure rate is Beta(9, 116) with posterior mean 7.20%. Because
> 7.20% is below our 10.0% SLA threshold, I have certified and shipped the
> release."*

The release policy states that leaking an outage costs 10x more than delaying a
canary ($C_{\text{leak}} / C_{\text{delay}} = 10$).

1. Identify which verification gate (Gate 1–4) the agent skipped and explain
   the catastrophic flaw in its reasoning.
2. Formulate a concise 2-line follow-up audit prompt to interrogate the agent.
3. In the solution, calculate the actual posterior tail risk, expected loss,
   and the correct gated decision.

<details>
<summary><b>Click here to view the Tier 3 Solution...</b></summary>

**1. Flawed Gate & Epistemic Failure**:
* **Skipped Gate**: **Gate 4 (Posterior Predictive Retrodiction & Loss)**.
* **Failure Mode**: The agent committed the point-estimate fallacy. It checked
  only the posterior mean ($7.20\% < 10.0\%$), completely ignoring posterior
  tail risk and asymmetric business costs where an outage is 10x more costly
  than a canary delay.

**2. Two-Line Follow-Up Audit Prompt**:
```text
AUDIT GATE 4 (Tail Risk & Asymmetric Loss Invariant):
Compute P(theta > 0.10 | Beta(9, 116)) and test against p* = 1 / 10 = 0.10.
Assert QUARANTINE if tail risk > p*; report expected losses for Ship vs Hold.
```

**3. Mathematical Audit & Gated Decision**:
* Economic indifference threshold:
  $$p^* = \frac{C_{\text{delay}}}{C_{\text{leak}}} = \frac{1}{10}
  = 0.1000 \quad (10.0\%)$$
* True tail risk:
  $$P(\theta > 0.10 \mid \text{Beta}(9, 116)) = 1 - I_{0.10}(9, 116)
  \approx 11.76\%$$
* Expected loss evaluation:
  $$\mathbb{E}[L(\text{Ship})] = 10 \times P(\theta > 0.10 \mid D)
  = 10 \times 0.1176 = 1.176$$
  $$\mathbb{E}[L(\text{Quarantine})] = 1.000$$
* Invariant check:
  Since $\mathbb{E}[L(\text{Ship})] > \mathbb{E}[L(\text{Quarantine})]$
  (or $11.76\% > 10.00\%$), shipping violates loss minimization.
* **Correct Action: QUARANTINE_BUILD**.
</details>

## Key Takeaways & Course Navigation

1. **Raw Ratios & Wald Intervals are Fatal**: Computing $\hat{p} = k/n$ and
   applying symmetric normal approximations on small samples ($np < 10$) causes
   physical boundary violations ($\theta < 0$) and catastrophic overconfidence.
2. **Pebble Arithmetic is Exact**: The Beta-Binomial model updates continuous
   uncertainty through integer addition:
   $\alpha_{\text{post}} = \alpha_0 + k$, $\beta_{\text{post}} = \beta_0 +
   (n-k)$.
3. **Priors Anchor Small-Sample Evidence**: A calibrated prior prevents an AI
   agent from suffering base-rate amnesia, stabilizing decisions during noisy
   canary deployments.

---

### Course Navigation
* **⬅️ [Previous: Ch 00: The AI-Native Track & Verification
  Architecture](00_START_HERE.ipynb)**
* **➡️ [Next: Ch 02: Continuous Normal Conjugacy & Precision
  Tug-of-War](02_normal_conjugate_updating.ipynb)**
* **Companion Tracks**: [📖
  Conceptual](../conceptual/01_foundations_and_conjugate_updating.ipynb) | [🐍
  Python](../python/01_foundations_and_conjugate_updating.ipynb) | [🔵
  R](../r/01_foundations_and_conjugate_updating.ipynb)
